# PASSO 3 - Treinamento de Modelo (XGBoost)
Nesse passo, vamos treinar modelos XGBoost para cada traço de personalidade MBTI.

## 0. Setup
Importar os pacotes necessários para rodar o resto do código.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score
from pipeline.smote_oversampling import SMOTEOversampler
from pipeline.machine_learning_algorithms.xgboost import XGBoostClassifier

# Tamanho do subset humano original no passo 2 (antes da augmentação por SLM).
# Validado em cada experimento "base" para garantir que o fatiamento [:N_HUMAN_POSTS]
# realmente corresponde aos posts humanos e não inclui amostras augmentadas.
N_HUMAN_POSTS = 6940
DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "train_datasets"
TEST_DIR = DATA_DIR / "test_datasets"
MODELS_DIR = Path("models/xgboost")
XGB_PARAMS = dict(
    n_estimators=800, 
    learning_rate=0.03, 
    max_depth=12, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    random_state=42
)

def divide_mbti(y_series):
    """Converte a string MBTI (ex: 'INFJ') em 4 colunas binárias (I/E, N/S, T/F, J/P)."""
    y_df = pd.DataFrame()
    y_df['I_E'] = (y_series.str[0] == 'I').astype(int)
    y_df['N_S'] = (y_series.str[1] == 'N').astype(int)
    y_df['T_F'] = (y_series.str[2] == 'T').astype(int)
    y_df['J_P'] = (y_series.str[3] == 'J').astype(int)
    return y_df

def load_train_test(embedding):
    """Carrega x_train/x_test (para um embedding) e y_train/y_test (comuns a todos)."""
    x_train = pd.read_pickle(TRAIN_DIR / f"x_train_{embedding}.pkl")
    x_test = pd.read_pickle(TEST_DIR / f"x_test_{embedding}.pkl")
    y_train = pd.Series(pd.read_pickle(TRAIN_DIR / "y_train.pkl"))
    y_test = pd.Series(pd.read_pickle(TEST_DIR / "y_test.pkl"))
    return x_train, x_test, y_train, y_test

def save_models(clf, label):
    """Salva os 4 submodelos (um por traço MBTI) com sufixo `label`."""
    for key, model in clf.models.items():
        path = MODELS_DIR / f"{key}_{label}.json"
        model.save_model(str(path))
        print(f"Salvo: {path}")

def run_experiment(embedding, label, *, use_smote=True, human_only=False):
    """Carrega dados, (opcionalmente) aplica SMOTE, treina, avalia e salva os modelos.

    - `human_only=True`: restringe o treino aos posts humanos originais (sem augmentação SLM).
    - `use_smote=False`: não aplica oversampling (baseline sem SMOTE).
    Retorna o classificador treinado (predições ficam disponíveis via clf.predict)."""
    x_train, x_test, y_train, y_test = load_train_test(embedding)

    if human_only:
        assert len(x_train) >= N_HUMAN_POSTS, (
            f"x_train ({len(x_train)}) menor que N_HUMAN_POSTS ({N_HUMAN_POSTS}); "
            "verifique a ordem de concatenação no passo 2."
        )
        x_train = x_train[:N_HUMAN_POSTS]
        y_train = y_train[:N_HUMAN_POSTS]

    if use_smote:
        x_train, y_train = SMOTEOversampler(random_state=42).fit_resample(x_train, y_train)

    y_train_4cols = divide_mbti(y_train)
    y_test_4cols = divide_mbti(y_test.reset_index(drop=True))

    clf = XGBoostClassifier(**XGB_PARAMS)
    clf.fit(x_train, y_train_4cols)
    clf.evaluate(x_test, y_test_4cols)
    save_models(clf, label)
    return clf

## 1. Baseline
Treina apenas com os posts humanos originais, sem aplicação de técnicas de balanceamento de classes. Serve como parâmetro de controle para avaliar o ganho das otimizações posteriores.

#### TF-IDF

In [ ]:
tfidf_baseline = run_experiment('tfidf', 'tfidf_baseline', use_smote=False, human_only=True)

#### Word2Vec

In [ ]:
w2v_baseline = run_experiment('w2v', 'w2v_baseline', use_smote=False, human_only=True)

## 2. Baseline + SMOTE
Treina com um over-sampling SMOTE dos dados originais. Gerando amostras sintéticas baseadas nas classes minoritárias, visando reduzir o viés do modelo.

#### TF-IDF

In [ ]:
tfidf_smote = run_experiment('tfidf', 'tfidf_smote', use_smote=True, human_only=True)

#### Word2Vec

In [ ]:
w2v_smote = run_experiment('w2v', 'w2v_smote', use_smote=True, human_only=True)

## 3. Baseline + SMOTE + SLM
Treina com um over-sampling e um augmentation de SLM (qwen) dos dados originais. O objetivo é testar se a variabilidade introduzida pela IA melhora a capacidade de generalização do modelo.

#### TF-IDF

In [ ]:
tfidf_aug = run_experiment('tfidf', 'tfidf_aug', use_smote=True, human_only=False)

#### Word2Vec

In [ ]:
w2v_aug = run_experiment('w2v', 'w2v_aug', use_smote=True, human_only=False)

## 4. Análise Comparativa
Avaliação da performance preditiva dos modelos, comparando os resultados das predições de cada experimento.

In [ ]:
x_test_tfidf = pd.read_pickle(TEST_DIR / "x_test_tfidf.pkl")
x_test_w2v = pd.read_pickle(TEST_DIR / "x_test_w2v.pkl")
y_test_raw = pd.Series(pd.read_pickle(TEST_DIR / "y_test.pkl"))
y_test_4cols = divide_mbti(y_test_raw)

pred_tfidf_base = tfidf_baseline.predict(x_test_tfidf)
pred_w2v_base = w2v_baseline.predict(x_test_w2v)
pred_tfidf_smote = tfidf_smote.predict(x_test_tfidf)
pred_w2v_smote = w2v_smote.predict(x_test_w2v)
pred_tfidf_aug = tfidf_aug.predict(x_test_tfidf)
pred_w2v_aug = w2v_aug.predict(x_test_w2v)

colunas = ['I_E', 'N_S', 'T_F', 'J_P']
nomes_eixos = ['I/E\n(Introvert/Extravert)', 'N/S\n(Intuition/Sensing)',
               'T/F\n(Thinking/Feeling)', 'J/P\n(Judging/Perceiving)']

experiments = [
    ('1. Base (No SMOTE)', 'TF-IDF', pred_tfidf_base),
    ('2. SMOTE Only',      'TF-IDF', pred_tfidf_smote),
    ('3. SLM Aug + SMOTE', 'TF-IDF', pred_tfidf_aug),
    ('1. Base (No SMOTE)', 'Word2Vec', pred_w2v_base),
    ('2. SMOTE Only',      'Word2Vec', pred_w2v_smote),
    ('3. SLM Aug + SMOTE', 'Word2Vec', pred_w2v_aug),
]

dados = [
    {'Axis': nomes_eixos[i], 'F1 (%)': f1_score(y_test_4cols[col], pred[col], average='weighted') * 100,
     'Method': method, 'Embedding': emb}
    for method, emb, pred in experiments
    for i, col in enumerate(colunas)
]
df_plot = pd.DataFrame(dados)

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)
sns.set_theme(style="whitegrid")
paleta = ['#e74c3c', '#f39c12', '#2ecc71']

for ax, emb in zip(axes, ['TF-IDF', 'Word2Vec']):
    sns.barplot(data=df_plot[df_plot['Embedding'] == emb],
                x='Axis', y='F1 (%)', hue='Method', palette=paleta, ax=ax)
    ax.set_title(f"{emb}: Impact of SMOTE & Augmentation",
                 fontsize=15, fontweight='bold', pad=15)
    ax.set_ylim(0, 115)
    ax.set_xlabel("")
    if emb == 'TF-IDF':
        ax.set_ylabel("Weighted F1-Score (%)", fontsize=13)
    else:
        ax.set_ylabel("")
    for p in ax.patches:
        altura = p.get_height()
        if altura > 0:
            ax.annotate(f"{altura:.1f}%",
                        (p.get_x() + p.get_width() / 2., altura),
                        ha='center', va='baseline', fontsize=11,
                        fontweight='bold', color='black',
                        xytext=(0, 6), textcoords='offset points')
    ax.legend(title='Experiment Stage', fontsize='11', loc='upper right')

sns.despine(left=True, bottom=False)
plt.tight_layout()
plt.show()